In [ ]:
import json
from pathlib import Path
import seaborn as sns
from collections import Counter, defaultdict
from tqdm.cli import tqdm
import difflib
from itertools import zip_longest
import random

In [ ]:
from sys import path as sys_path
project_root = Path.cwd().parent.parent
if project_root not in sys_path:
    sys_path.append(str(project_root))
data_dir = project_root/"data/"
outputs_dir = project_root/"outputs/"

 ## Process JSONL files

In [ ]:
migration_data_directory = Path(outputs_dir / "json/migration_jsonls/2-final")
files = sorted(migration_data_directory.glob("*_migration_events.jsonl"))

In [ ]:
random.seed = 42

In [ ]:
def page_key(d):
    """page id from a dict with book, page_n and side."""
    return (d["book"], d["page_n"], d["side"])

pages = defaultdict(list)

random_files = random.sample(files, 200)
files_with_duplicates = []
for file in tqdm(random_files):
    with open(file, "r", encoding="utf-8") as f:
        records = [json.loads(line) for line in f]

    if not any(record["duplicate"] for record in records):
        continue

    files_with_duplicates.append(file)
    for record in records:
        pages[page_key(record)].append(record)

random_files = files_with_duplicates

In [ ]:
duplicates = {}

for key, events in pages.items():
    dup_keys = set()
    for event in events:
        for dup in event.get("duplicate_of") or []:
            dup_keys.add(page_key(dup))
    if dup_keys:
        duplicates[key] = dup_keys

print(f"Pages loaded: {len(pages)}")
print(f"Pages with duplicates: {len(duplicates)}")

In [ ]:
def page_lines(key):
    """One string per event on the page, with row_data joined by ", "."""
    return [", ".join(map(str, event["row_data"])) for event in pages[key]]

def side_by_side(left, right, width=60):
    """Render two lists of lines in two columns. Markers: | changed, > only left, < only right."""
    rows = []
    matcher = difflib.SequenceMatcher(None, left, right)
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        for a, b in zip_longest(left[i1:i2], right[j1:j2], fillvalue=""):
            if tag == "equal":
                marker = " "
            elif a and b:
                marker = "|"
            elif a:
                marker = ">"
            else:
                marker = "<"
            rows.append(f"{a[:width]:<{width}} {marker} {b[:width]}")
    return "\n".join(rows)

def compare_random_pages(n, width=80):
    sampled = random.sample(list(duplicates.keys()), min(n, len(duplicates)))

    for key in sampled:
        original = page_lines(key)
        for dup_key in duplicates[key]:
            print(f"=== ORIGINAL {key} vs DUPLICATE {dup_key} ===")
            if dup_key not in pages:
                print("page not found in loaded files\n")
                continue

            duplicate = page_lines(dup_key)
            print(f"{'ORIGINAL':<{width}}   {'DUPLICATE'}")
            print(side_by_side(original, duplicate, width=width) or "identical", "\n")

compare_random_pages(n=100)